## Assignment 1
* Name: Oluwafemi Oladosu
* StudentID: ARI/2026/TC-7/0072
* A mini data_cleaning function codes

##### Please, note that Gemini AI was consulted to debug some errors during this project

### Brief documentation

##### This `function` performs a quick cleaning on any messy data.
Below are the cleaning process this function will perform on your messy data
- install the neccesary libraries
- create a copy to work on
- strip white space from all your entries
- standardise all missing values to the format recognised by np
- removing duplicate is quite tricky, if you wanna remove it, just uncomment the reove duplicate section
- normalise all datatype, and compare the dtype before and after cleaning
- if you have a name column, it will normalise it, and also correct the short form if there is a duplicate of the fullname
- if you have email column, it will normalise it and fill missing based on name.
- identify numeric columns and detect outliers using IQR method
- fill missing values in numeric columns with median
- drop remaining missing values in non-numeric columns
- print the summary
- Save the clean data as clean_data.csv
##### Usage
To use this function, simply call it with your DataFrame, example below:

`df = pd.read_csv("messy_dataset_50.csv")`

`clean_df= generalPurposeCleaning(df)`

* if you are using colab, just run the mount cell



In [6]:
import pandas as pd
import numpy as np
import re
from difflib import get_close_matches
from pandas.api.types import is_string_dtype
import openpyxl

In [22]:

def generalPurposeCleaning(df, id_keywords=['id'], iqr_multiplier=1.5, threshold=0.5):

    # installation of necessary libraries
    import pandas as pd
    import numpy as np
    import re
    from difflib import get_close_matches
    from pandas.api.types import is_string_dtype
    import openpyxl
     # ----------------------------------------------------------------------------------------------------------------------------

    df = df.copy() # this is to avoid modifying the original dataframe outside the function, we work on a copy of it.

    # Removal of Strip whitespace
    for col in df.columns:
        df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)

    # ----------------------------------------------------------------------------------------------------------------------------

    # standardise missing values
    df = df.replace(['Nan', 'NaN', 'nan', '', 'None', 'null'], np.nan)

    # Remove duplicates
    #df = df.drop_duplicates()
    # ----------------------------------------------------------------------------------------------------------------------------

    # Normalisation of column names
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(' ', '_')
    )
    # ----------------------------------------------------------------------------------------------------------------------------

    # Standardisation of string columns (except emails)
    for col in df.columns:
        if is_string_dtype(df[col]) and 'email' not in col.lower():
            df[col] = df[col].apply(
                lambda x: (
                    x.title() if isinstance(x, str) and len(x.strip()) > 3
                    else x.upper() if isinstance(x, str) and len(x.strip()) <= 3 and x.islower()
                    else x
                )
            )
    # ----------------------------------------------------------------------------------------------------------------------------
    # convert data types
    dtype_before = df.dtypes.copy()  # this is for comparison later

    # This loop through object/string columns to detect if they are actually dates or numeric values stored as strings,
    # and convert them accordingly.
    for col in df.select_dtypes(include=['object', 'string']):
        sample = df[col].dropna().astype(str)
        if sample.empty:
            continue

       # this regex checks for common date formats like YYYY-MM-DD, DD/MM/YYYY, MM.DD.YYYY etc.
       # if more than 60% of the non-null values match this pattern, we convert the column to datetime.
        if sample.str.contains(r'\d{1,4}[-/\.]\d{1,2}[-/\.]\d{1,4}').mean() > 0.6:
            df[col] = pd.to_datetime(df[col], errors='coerce', format='mixed', dayfirst=True)
            continue

        # this part is for cleaning numeric values that are stored as strings,
        # it removes common non-numeric characters and words like "years", then tries to convert to numeric.
        cleaned = (
            df[col]
            .astype(str)
            .str.replace(',', '', regex=False)
            .str.replace(r'\s*years?', '', regex=True)
            .str.strip()
        )

      # this works on irrelevant upper and lower case string values
        if 'email' in col.lower():
            continue
        df[col] = df[col].apply(
            lambda x: (
                x.strip().title() if isinstance(x, str) and len(x.strip()) > 3
                else x.strip().upper() if isinstance(x, str) and len(x.strip()) <= 3
                else x
            )
        )

       # this converts the cleaned string to numeric, converting errors to NaN.
       # If we get a significant number of valid numeric values, we keep the conversion.
        numeric_col = pd.to_numeric(cleaned, errors='coerce')
        if numeric_col.notna().sum() > 0:
            df[col] = numeric_col



    # ---------------------------------------------------------------------------------------------------------------------------

    # normalising name and fixing incrorect names that are duplicate
    if 'name' in df.columns:

        df['name'] = df['name'].astype(str).str.strip()
        df['name'] = df['name'].str.replace(r'[^a-zA-Z\s]', '', regex=True)
        #df['name'] = df['name'].str.title()
        #df['name'] = df['name'].replace(['', 'Nan'], np.nan)
        df['name'] = df['name'].fillna('Unknown')

        unique_names = df['name'].dropna().unique().tolist()
        name_mapping = {}
        for name in unique_names:
            possible_matches = [n for n in unique_names if n != name]
            matches = get_close_matches(name, possible_matches, n=1, cutoff=0.7)
            if matches:
                best_match = matches[0]
                if len(best_match) > len(name):
                    name_mapping[name] = best_match
        df['name'] = df['name'].replace(name_mapping)  # this replace names with their best match

     # ----------------------------------------------------------------------------------------------------------------------------

    # Normalising email and filling missing emails based on name
    if 'email' in df.columns:

        email_pattern = r'^[\w\.-]+@[\w\.-]+\.\w+$' #P proper email pattern
        def validate_email(x):
            if pd.isna(x):
                return np.nan
            x = str(x).strip().lower()
            if re.match(email_pattern, x):
                return x
            else:
                return np.nan
                # Apply validation
        df['email'] = df['email'].apply(validate_email)

        if 'name' in df.columns and 'email' in df.columns:

            # creating an email lookup based on name
            email_map = (
                df.dropna(subset=['email'])   # keep rows with valid email
                .groupby('name')['email']
                .first()                   # take first valid email for each name
                .to_dict()
            )

            # filling missing emails based on name
            df['email'] = df.apply(
                lambda row: email_map.get(row['name'], np.nan)
                if pd.isna(row['email']) else row['email'],
                axis=1
            )

        # Detect most common domain from valid emails
        valid_emails = df['email'].dropna()
        if not valid_emails.empty:
            domain = valid_emails.str.split('@').str[1].mode()[0]
        else:
            domain = "example.com"  # fallback

        # generate emails for rows with missing email but known name
        df['email'] = df.apply(
            lambda row: f"{row['name'].replace(' ', '').lower()}@{domain}"
            if pd.isna(row['email']) and row['name'] != 'Unknown'
            else row['email'],
            axis=1
        )

     # ----------------------------------------------------------------------------------------------------------------------------

    # Identify numeric columns and detect outliers using IQR method
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

    numeric_cols = [
        col for col in numeric_cols
        if not any(keyword in col.lower() for keyword in id_keywords)
    ]
    # detect outliers using IQR method
    mask = pd.DataFrame(True, index=df.index, columns=numeric_cols)
    outlier_counts = {}

    for col in numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - iqr_multiplier * IQR
        upper = Q3 + iqr_multiplier * IQR

        col_mask = df[col].between(lower, upper, inclusive='both') | df[col].isna()

        # Count real outliers (exclude NaNs)
        outlier_counts[col] = ((~col_mask) & df[col].notna()).sum()

        mask[col] = col_mask

    row_score = mask.sum(axis=1) # this counts how many numeric columns are non-outliers for each row

    cleaned_df = df[row_score >= len(numeric_cols) - 1].copy()
    rows_after_outliers = len(cleaned_df) # this is for summary report
     # ----------------------------------------------------------------------------------------------------------------------------

    # Filling missing values in numeric columns with median
    fill_report = {}

    for col in numeric_cols:
        median_value = cleaned_df[col].median()
        missing_count = cleaned_df[col].isna().sum()

        cleaned_df[col] = cleaned_df[col].fillna(median_value)

        fill_report[col] = {
            "filled_values": int(missing_count),
            "median_used": float(median_value) if pd.notna(median_value) else None
        }
     # ----------------------------------------------------------------------------------------------------------------------------

    # Dropping remaining NaNs in non-numeric columns
    rows_before_dropna = len(cleaned_df)
    cleaned_df = cleaned_df.dropna().reset_index(drop=True)
    rows_after_dropna = len(cleaned_df)
     # ----------------------------------------------------------------------------------------------------------------------------


    # This is the summary report that will be printed at the end, it includes all the key metrics and changes that happened during the cleaning process.
    # It also includes a comparison of data types before and after cleaning to show any changes that were made.
    summary = {
        "numeric_columns_used": len(numeric_cols),
        "columns_checked": numeric_cols,
        "outliers_detected_per_column": outlier_counts,
        "rows_after_outlier_filter": rows_after_outliers,
        "rows_removed_due_to_remaining_nan": rows_before_dropna - rows_after_dropna,
        "rows_final": rows_after_dropna,
        "fill_report": fill_report,
        "final_shape": cleaned_df.shape
    }
    metrics = {
        'Metric': [
            'Numeric Columns Used',
            'Rows After Outlier Filter',
            'Rows Removed (NaN)',
            'Final Rows',
            'Final Shape'
        ],
        'Value': [
            summary['numeric_columns_used'],
            summary['rows_after_outlier_filter'],
            summary['rows_removed_due_to_remaining_nan'],
            summary['rows_final'],
            summary['final_shape']
        ]
    }
    metrics_df = pd.DataFrame(metrics)

    # Data type comparison before and after cleaning
    dtype_after = cleaned_df.dtypes.copy()

    dtype_comparison = pd.DataFrame({
        'Column': dtype_before.index,
        'Before': dtype_before.values,
        'After': dtype_after.reindex(dtype_before.index).values
    })


    dtype_comparison['Changed'] = dtype_comparison['Before'] != dtype_comparison['After']


    cleaned_df.to_csv("cleaned_data.csv", index=False) # this saves the cleaned DataFrame to a new CSV file called "cleaned_data.csv"
    cleaned_df.to_csv("/content/drive/MyDrive/Tech crush_AI_ML/Assignment/cleaned_data.csv", index =False)
    print("Cleaned data exported as 'cleaned_data.csv' into your root folder")
    print("")

    print("------------------------Data Type Comparison Before vs After Cleaning-------------------------")
    print(dtype_comparison)

    print("")
    print("------------------------Cleaning Summary--------------------------------------------------")
    print(metrics_df)
    print("")
    print("-------------------------First 5 rows of cleaned data--------------------------------------")
    print(cleaned_df.head())
     # ----------------------------------------------------------------------------------------------------------------------------

    return cleaned_df

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
df = pd.read_csv("/content/drive/MyDrive/Tech crush_AI_ML/Assignment/messy_dataset_50.csv")

clean_df= generalPurposeCleaning(df)


Cleaned data exported as 'cleaned_data.csv' into your root folder

------------------------Data Type Comparison Before vs After Cleaning-------------------------
       Column  Before           After  Changed
0          id   int64           int64    False
1        name  object          object    False
2         age  object         float64     True
3       email  object          object    False
4   join_date  object  datetime64[ns]     True
5      salary  object         float64     True
6  department  object          object    False

------------------------Cleaning Summary--------------------------------------------------
                      Metric    Value
0       Numeric Columns Used        2
1  Rows After Outlier Filter       50
2         Rows Removed (NaN)       31
3                 Final Rows       19
4                Final Shape  (19, 7)

-------------------------First 5 rows of cleaned data--------------------------------------
   id    name   age            email  join_date  

In [20]:
df = pd.read_excel("/content/drive/MyDrive/Tech crush_AI_ML/Assignment/Superstore Sales2.xlsx")

clean_sales_df= generalPurposeCleaning(df)

Cleaned data exported as 'cleaned_data.csv' into your root folder

------------------------Data Type Comparison Before vs After Cleaning-------------------------
           Column          Before           After  Changed
0          row_id           int64           int64    False
1        order_id          object          object    False
2      order_date  datetime64[ns]  datetime64[ns]    False
3       ship_date  datetime64[ns]  datetime64[ns]    False
4       ship_mode          object          object    False
5     customer_id          object          object    False
6   customer_name          object          object    False
7         segment          object          object    False
8         country          object          object    False
9            city          object          object    False
10          state          object          object    False
11    postal_code           int64           int64    False
12         region          object          object    False
13     produ